 1.Importing the Dependencies

In [ ]:
# Data manipulation
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Date/time handling
import datetime as dt

# Ignore warnings (optional)
import warnings
warnings.filterwarnings("ignore")


2. Data Loading and Understanding

In [ ]:

import pandas as pd

# Chicago Crime dataset - pull from City of Chicago API
# Filtering: crimes from 2019 onwards (~5 years of data)
url = "https://data.cityofchicago.org/resource/ijzp-q8t2.csv?$where=date>'2019-01-01T00:00:00'&$limit=500000"

# Read into pandas
df = pd.read_csv(url)

print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())
df.head()



In [ ]:
# Convert 'date' column to datetime
df['date'] = pd.to_datetime(df['date'])

# Extract features
df['hour'] = df['date'].dt.hour
df['day_of_week'] = df['date'].dt.day_name()
df['month'] = df['date'].dt.month
df['year'] = df['date'].dt.year

# Quick check
df[['date', 'hour', 'day_of_week', 'month', 'year']].head()


Missing Values Overview

In [ ]:
df.isnull().sum().sort_values(ascending=False).head(10)


Exploratory Data Analysis (EDA)

In [ ]:
plt.figure(figsize=(12,6))
sns.countplot(data=df, y='primary_type', order=df['primary_type'].value_counts().head(10).index)
plt.title("Top 10 Most Common Crime Types (2019–2025)")
plt.xlabel("Count")
plt.ylabel("Crime Type")
plt.show()


Define a “Hotspot”

In [ ]:
# Count crimes by community area
crime_counts = df['community_area'].value_counts()

# Define hotspots as top 10% areas
threshold = crime_counts.quantile(0.90)
hotspots = crime_counts[crime_counts >= threshold].index

# Create hotspot label (1 = hotspot, 0 = not)
df['hotspot'] = df['community_area'].apply(lambda x: 1 if x in hotspots else 0)

df[['community_area','hotspot']].head()


In [ ]:
# Aggregate daily crime counts
daily_counts = df.groupby('date').size().reset_index(name='crime_count')
daily_counts.head()


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12,6))
plt.plot(daily_counts['date'], daily_counts['crime_count'])
plt.title("Daily Crime Counts in Chicago (2019–Present)")
plt.xlabel("Date")
plt.ylabel("Number of Crimes")
plt.show()


In [ ]:
df['week'] = pd.to_datetime(df['date']).dt.to_period('W')
weekly_counts = df.groupby('week').size().reset_index(name='crime_count')

plt.figure(figsize=(12,6))
plt.plot(weekly_counts['week'].astype(str), weekly_counts['crime_count'])
plt.title("Weekly Crime Trends")
plt.xlabel("Week")
plt.ylabel("Crimes")
plt.xticks(rotation=45)
plt.show()


Overview of the Data

In [ ]:
# Basic info
df.info()

# Summary statistics
df.describe()

# Check for missing values
df.isnull().sum().sort_values(ascending=False)


Distribution of Crimes Over Time

In [ ]:
# Crimes per year
df['year'] = pd.to_datetime(df['date']).dt.year
yearly_counts = df['year'].value_counts().sort_index()
plt.figure(figsize=(10,5))
sns.lineplot(x=yearly_counts.index, y=yearly_counts.values)
plt.title("Crimes per Year in Chicago (2019–Present)")
plt.xlabel("Year")
plt.ylabel("Number of Crimes")
plt.show()

# Crimes per month
df['month'] = pd.to_datetime(df['date']).dt.month
monthly_counts = df['month'].value_counts().sort_index()
sns.barplot(x=monthly_counts.index, y=monthly_counts.values)
plt.title("Crimes by Month")
plt.xlabel("Month")
plt.ylabel("Number of Crimes")
plt.show()


Top Crime Types

In [ ]:
plt.figure(figsize=(10,6))
df['primary_type'].value_counts().head(10).plot(kind='bar')
plt.title("Top 10 Crime Types in Chicago (2019–Present)")
plt.xlabel("Crime Type")
plt.ylabel("Count")
plt.xticks(rotation=45)
plt.show()


Crimes by Location Type

In [ ]:
plt.figure(figsize=(10,6))
df['location_description'].value_counts().head(10).plot(kind='barh')
plt.title("Top 10 Locations of Crime Occurrence")
plt.xlabel("Number of Crimes")
plt.ylabel("Location Type")
plt.show()




In [ ]:
sns.kdeplot(
    data=df.sample(10000),
    x='longitude',
    y='latitude',
    fill=True,
    cmap='Reds',
    alpha=0.6
)
plt.title("Crime Density Across Chicago")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.show()


Arrest and Domestic Patterns

In [ ]:
# Arrest ratio
plt.figure(figsize=(5,5))
df['arrest'].value_counts().plot(kind='pie', autopct='%1.1f%%', startangle=90)
plt.title("Arrest vs Non-Arrest Cases")
plt.ylabel("")
plt.show()

# Domestic incidents
plt.figure(figsize=(5,5))
df['domestic'].value_counts().plot(kind='pie', autopct='%1.1f%%', startangle=90)
plt.title("Domestic vs Non-Domestic Crimes")
plt.ylabel("")
plt.show()


Crimes by Community Area

In [ ]:
plt.figure(figsize=(12,6))
df['community_area'].value_counts().head(15).plot(kind='bar')
plt.title("Top 15 Community Areas by Crime Count")
plt.xlabel("Community Area")
plt.ylabel("Number of Crimes")
plt.show()


Temporal Trends by Day and Hour

In [ ]:
df['day_of_week'] = pd.to_datetime(df['date']).dt.day_name()
df['hour'] = pd.to_datetime(df['date']).dt.hour

# Day of week
plt.figure(figsize=(8,5))
sns.countplot(x='day_of_week', data=df, order=['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday'])
plt.title("Crimes by Day of the Week")
plt.xticks(rotation=45)
plt.show()

# Hour of day
plt.figure(figsize=(10,5))
sns.countplot(x='hour', data=df)
plt.title("Crimes by Hour of the Day")
plt.xlabel("Hour")
plt.ylabel("Count")
plt.show()


Correlation Heatmap (for Numeric Columns)

In [ ]:
plt.figure(figsize=(10,6))
sns.heatmap(df.corr(numeric_only=True), annot=True, cmap='coolwarm')
plt.title("Correlation Between Numerical Features")
plt.show()


Geospatial Visualization

In [ ]:
plt.figure(figsize=(8,6))
sns.scatterplot(x='longitude', y='latitude', data=df.sample(10000), alpha=0.3)
plt.title("Crime Locations in Chicago")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.show()


### Key Insights from EDA
- Most crimes occur in [Top 3 areas].
- Crime rates were highest in [Year] and decreased in [Year].
- Theft and battery are the most frequent crime types.
- Street and residence are the most common crime locations.
- Spatial density maps show high concentration in central and south Chicago regions.
- This exploratory analysis will guide the hotspot and forecasting models in the next phase.


In [ ]:
# Build classification dataframe
clf_df = df[[
    'primary_type',
    'location_description',
    'arrest',
    'domestic',
    'community_area',
    'hour',
    'day_of_week',
    'month',
    'hotspot'
]].dropna()

print("Shape:", clf_df.shape)
clf_df.head()


In [ ]:
from sklearn.preprocessing import LabelEncoder

categoricals = ['primary_type', 'location_description', 'day_of_week']

le = LabelEncoder()
for col in categoricals:
    clf_df[col] = le.fit_transform(clf_df[col])


In [ ]:
from sklearn.model_selection import train_test_split

X = clf_df.drop('hotspot', axis=1)
y = clf_df['hotspot']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)


In [ ]:
from sklearn.linear_model import LogisticRegression

log_reg = LogisticRegression(max_iter=500)
log_reg.fit(X_train, y_train)

y_pred_lr = log_reg.predict(X_test)


In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=20,
    random_state=42
)

rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)


In [ ]:
from xgboost import XGBClassifier

xgb = XGBClassifier(
    n_estimators=400,
    learning_rate=0.05,
    max_depth=6,
    random_state=42
)

xgb.fit(X_train, y_train)
y_pred_xgb = xgb.predict(X_test)


In [ ]:
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

def evaluate_model(name, y_test, y_pred):
    print(f"\n{name} Results:")
    print("Accuracy:", accuracy_score(y_test, y_pred))
    print("F1 Score:", f1_score(y_test, y_pred))
    print("\nClassification Report:\n", classification_report(y_test, y_pred))
    sns.heatmap(confusion_matrix(y_test, y_pred), annot=True, fmt='d')
    plt.title(f"Confusion Matrix - {name}")
    plt.show()

evaluate_model("Logistic Regression", y_test, y_pred_lr)
evaluate_model("Random Forest", y_test, y_pred_rf)
evaluate_model("XGBoost", y_test, y_pred_xgb)


In [ ]:
importances = pd.Series(rf.feature_importances_, index=X.columns)
importances.sort_values().plot(kind='barh', figsize=(10,6))
plt.title("Feature Importance (Random Forest)")
plt.show()


In [ ]:
B

In [ ]:
daily = df.groupby(df['date'].dt.date).size().reset_index(name='crime_count')
daily['date'] = pd.to_datetime(daily['date'])
daily.set_index('date', inplace=True)
daily = daily.asfreq('D').fillna(0)   # fill missing days with 0 crimes
daily.head()


In [ ]:
import statsmodels.api as sm

sarima = sm.tsa.statespace.SARIMAX(
    daily['crime_count'],
    order=(2,1,2),
    seasonal_order=(1,1,1,7)  # weekly seasonality
).fit()

sarima.summary()


In [ ]:
sarima_forecast = sarima.get_forecast(steps=90)
sarima_pred = sarima_forecast.predicted_mean

plt.figure(figsize=(12,6))
plt.plot(daily.index, daily['crime_count'], label='Actual')
plt.plot(sarima_pred.index, sarima_pred.values, label='SARIMA Forecast')
plt.legend()
plt.show()


In [ ]:
from prophet import Prophet

pf = daily.reset_index()
pf.columns = ['ds', 'y']


In [ ]:
model = Prophet(
    yearly_seasonality=True,
    weekly_seasonality=True,
    daily_seasonality=False
)

model.fit(pf)


In [ ]:
future = model.make_future_dataframe(periods=90)
forecast = model.predict(future)

model.plot(forecast)
plt.title("Prophet Forecast for Crime Count")
plt.show()

model.plot_components(forecast)
plt.show()
